In [1]:
# Fichier : notebooks/00_pretraitement.ipynb
# =============================================================================
# SCRIPT DE PRÉ-TRAITEMENT DES DONNÉES (À EXÉCUTER UNE SEULE FOIS)
# =============================================================================

# === CELLULE 1: IMPORTS ET CONFIGURATION DES CHEMINS ===
import os
import pickle
import pandas as pd
import numpy as np

print("--- Début du script de pré-traitement ---")
base_path = '..' 
data_path = os.path.join(base_path, 'data')
output_path = os.path.join(base_path, 'processed_data')
os.makedirs(output_path, exist_ok=True)
print(f"Les données traitées seront sauvegardées dans : {os.path.abspath(output_path)}")

# === CELLULE 2: CHARGEMENT, TRANSFORMATION ET SAUVEGARDE ===
try:
    # --- 1. CHARGEMENT DES FICHIERS EXCEL ---
    print("\n--- 1. Chargement des fichiers Excel depuis le dossier 'data/' ---")
    cours_path = os.path.join(data_path, 'all_datas.xlsx')
    dividendes_path = os.path.join(data_path, 'dividendes.xlsx')
    nb_actions_path = os.path.join(data_path, 'nb_actions.xlsx')
    actions_secteurs_pays_path = os.path.join(data_path, 'actions_secteurs_pays.xlsx')

    dividendes_df = pd.read_excel(dividendes_path)
    actions_secteurs_pays_df = pd.read_excel(actions_secteurs_pays_path)
    nb_actions_df = pd.read_excel(nb_actions_path)
    xls = pd.ExcelFile(cours_path)
    print("✅ Fichiers Excel chargés.")

    # --- 2. TRANSFORMATION DES DONNÉES DE PRIX (all_data) ---
    print("\n--- 2. Transformation des données de prix ('all_data') ---")
    all_data_list = []
    for act in xls.sheet_names:
        df = xls.parse(act).assign(Date=lambda x: pd.to_datetime(x.Date), Ticker=act)
        all_data_list.append(df)
    all_data_df = pd.concat(all_data_list, ignore_index=True)
    all_data_df = all_data_df.set_index(["Date", "Ticker"]).sort_index().unstack(level="Ticker")
    all_data_df.columns = all_data_df.columns.swaplevel(0, 1)
    all_data_df = all_data_df.sort_index(axis=1, level=0)
    all_data_df.columns.names = ["Ticker", "Price"]
    print(f"✅ 'all_data' transformé. Shape: {all_data_df.shape}")

    # --- 3. RENOMMAGE PRÉLIMINAIRE (selon votre logique originale) ---
    print("\n--- 3. Renommage des colonnes pour cohérence ---")
    nb_actions_df = nb_actions_df.rename(columns={"Symbol": "Ticker"})
    dividendes_df = dividendes_df.rename(columns={"Date": "Year"})
    print("✅ Colonnes renommées ('Symbol'->'Ticker', 'Date'->'Year').")

    # --- 4. CRÉATION DU PANEL DE DONNÉES FONDAMENTALES ---
    print("\n--- 4. Création du panel de données fondamentales ('fundamentals_panel') ---")
    nb_actions, actions_secteurs_pays, dividendes, all_data = nb_actions_df.copy(), actions_secteurs_pays_df.copy(), dividendes_df.copy(), all_data_df.copy()

    if 'Ticker' in nb_actions.columns: nb_actions.set_index('Ticker', inplace=True)
    if 'Ticker' in actions_secteurs_pays.columns: actions_secteurs_pays.set_index('Ticker', inplace=True)

    fundamentals = nb_actions.join(actions_secteurs_pays, how='left')
    if 'Company name' in fundamentals.columns: fundamentals = fundamentals.drop(columns=['Company name'])
    fundamentals_onehot = pd.get_dummies(fundamentals, columns=['Sector', 'Country'], dtype=float)

    dividendes_long = dividendes.melt(id_vars=['Year'], var_name='Ticker', value_name='Dividende').dropna(subset=['Year'])
    dividendes_long['Year'] = dividendes_long['Year'].astype(int)
    dividendes_pivot = dividendes_long.pivot_table(index='Ticker', columns='Year', values='Dividende')

    dates = all_data.index.get_level_values(0).unique()
    tickers = fundamentals.index
    panel_index = pd.MultiIndex.from_product([dates, tickers], names=['Date', 'Ticker'])
    fundamentals_panel = pd.DataFrame(index=panel_index)
    for col in fundamentals_onehot.columns:
        col_map = fundamentals_onehot[col].to_dict()
        fundamentals_panel[col] = fundamentals_panel.index.get_level_values('Ticker').map(col_map).fillna(0)
    
    fundamentals_panel = fundamentals_panel.astype(float).reset_index()
    fundamentals_panel['Year_prev'] = fundamentals_panel['Date'].dt.year - 1

    def get_dividende_vectorized(df, pivot):
        pivot_stacked = pivot.stack().reset_index(name='dividende_last_year')
        df = pd.merge(df, pivot_stacked, left_on=['Ticker', 'Year_prev'], right_on=['Ticker', 'Year'], how='left')
        df['dividende_last_year'] = df['dividende_last_year'].fillna(0.0).infer_objects(copy=False)
        return df.drop(columns=['Year'], errors='ignore')

    fundamentals_panel = get_dividende_vectorized(fundamentals_panel, dividendes_pivot)
    fundamentals_panel = fundamentals_panel.drop(columns=['Year_prev'], errors='ignore').set_index(['Date', 'Ticker'])
    print(f"✅ 'fundamentals_panel' créé. Shape: {fundamentals_panel.shape}")

    # --- 4.1. VALIDATION DES DONNÉES ---
    print("\n--- 4.1. Validation des données ---")
    print(f"  -> all_data: {all_data_df.shape} (dates: {all_data_df.index.nunique()}, actifs: {len(all_data_df.columns.get_level_values(0).unique())})")
    print(f"  -> fundamentals_panel: {fundamentals_panel.shape}")
    print(f"  -> Variables de prix par actif: {len(all_data_df.columns.get_level_values(1).unique())}")
    print(f"  -> Colonnes fondamentales: {list(fundamentals_panel.columns)}")

    # --- 5. SAUVEGARDE DE TOUS LES FICHIERS .PKL ---
    print("\n--- 5. Sauvegarde de tous les DataFrames traités ---")
    data_to_save = {
        'all_data.pkl': all_data_df,
        'nb_actions.pkl': nb_actions_df,
        'dividendes.pkl': dividendes_df,
        'actions_secteurs_pays.pkl': actions_secteurs_pays_df,
        'fundamentals_panel.pkl': fundamentals_panel
    }
    for fname, df_to_save in data_to_save.items():
        with open(os.path.join(output_path, fname), 'wb') as f: pickle.dump(df_to_save, f)
        print(f"  -> Fichier '{fname}' sauvegardé.")
        
    print("\n\n✅✅✅ SCRIPT DE PRÉ-TRAITEMENT TERMINÉ ! ✅✅✅")

except Exception as e:
    print(f"❌ ERREUR : {e}")
    raise e

--- Début du script de pré-traitement ---
Les données traitées seront sauvegardées dans : /Users/naofal/Desktop/Mémoire/mistral/processed_data

--- 1. Chargement des fichiers Excel depuis le dossier 'data/' ---
✅ Fichiers Excel chargés.

--- 2. Transformation des données de prix ('all_data') ---
✅ 'all_data' transformé. Shape: (6386, 225)

--- 3. Renommage des colonnes pour cohérence ---
✅ Colonnes renommées ('Symbol'->'Ticker', 'Date'->'Year').

--- 4. Création du panel de données fondamentales ('fundamentals_panel') ---
✅ 'fundamentals_panel' créé. Shape: (300142, 16)

--- 4.1. Validation des données ---
  -> all_data: (6386, 225) (dates: 6386, actifs: 45)
  -> fundamentals_panel: (300142, 16)
  -> Variables de prix par actif: 5
  -> Colonnes fondamentales: ['nb_actions', 'Sector_AGRICULTURE', 'Sector_DISTRIBUTION', 'Sector_FINANCE', 'Sector_INDUSTRY', 'Sector_OTHER', 'Sector_PUBLIC SERVICE', 'Sector_TRANSPORT', 'Country_BENIN', 'Country_BURKINA FASO', 'Country_IVORY COAST', 'Countr

/var/folders/p9/9fddyb8173q1lm0nmg7htmjc0000gn/T/ipykernel_51460/889206577.py:82: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['dividende_last_year'] = df['dividende_last_year'].fillna(0.0).infer_objects(copy=False)
